In [3]:
# Importação e configurações iniciais
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

sns.set_style('whitegrid')

In [4]:
# Importação de arquivos para junção
aisles = pd.read_csv('../data/aisles.csv')
departments = pd.read_csv('../data/departments.csv')
orders = pd.read_csv('../data/orders.csv')
order_products_prior = pd.read_csv('../data/order_products__prior.csv')
order_products_train = pd.read_csv('../data/order_products__train.csv')
products = pd.read_csv('../data/products.csv')

In [5]:
# Verificação de dimensões
datasets = {
    'aisles': aisles,
    'departments': departments,
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products
}

for nome, df in datasets.items():
    print(f'\n{nome}')
    print('-'*50)
    print(df.shape)


aisles
--------------------------------------------------
(134, 2)

departments
--------------------------------------------------
(21, 2)

orders
--------------------------------------------------
(3421083, 7)

order_products_prior
--------------------------------------------------
(32434489, 4)

order_products_train
--------------------------------------------------
(1384617, 4)

products
--------------------------------------------------
(49688, 4)


In [6]:
# Verificação de informações gerais dos arquivos
for nome, df in datasets.items():
    print(f'\n{"="*60}')
    print(nome.upper())
    print(df.info())


AISLES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   aisle_id  134 non-null    int64 
 1   aisle     134 non-null    object
dtypes: int64(1), object(1)
memory usage: 2.2+ KB
None

DEPARTMENTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   department_id  21 non-null     int64 
 1   department     21 non-null     object
dtypes: int64(1), object(1)
memory usage: 468.0+ bytes
None

ORDERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                object 
 3   order_number            

In [7]:
# Verificação de nulos
for nome, df in datasets.items():
    print(f'\n{nome}')
    print(df.isnull().sum())


aisles
aisle_id    0
aisle       0
dtype: int64

departments
department_id    0
department       0
dtype: int64

orders
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

order_products_prior
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

order_products_train
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

products
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64


Observação: days_since_prior_order são os dias desde a última compra, se o usuário só fez uma compra ou os dados só remetem à primeira compra do usuário, que é nosso caso, então não devemos nos preocupar com os nulos nessa coluna

In [8]:
# Verificando valores duplicados
for nome, df in datasets.items():
    print(f'{nome}: {df.duplicated().sum()} duplicatas')

aisles: 0 duplicatas
departments: 0 duplicatas
orders: 0 duplicatas
order_products_prior: 0 duplicatas
order_products_train: 0 duplicatas
products: 0 duplicatas


In [9]:
# Construindo dimensão de produtos
products_full = (
    products
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)

products_full.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [10]:
# Construindo tabela de pedidos anteriores
prior_full = (
    order_products_prior
    .merge(orders, on='order_id', how='left')
    .merge(products_full, on='product_id', how='left')
)

prior_full.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [11]:
# Otimização de memória como boa prática
prior_full['order_id'] = prior_full['order_id'].astype('int32')
prior_full['product_id'] = prior_full['product_id'].astype('int32')
prior_full['user_id'] = prior_full['user_id'].astype('int32')

prior_full['aisle'] = prior_full['aisle'].astype('category')
prior_full['department'] = prior_full['department'].astype('category')

In [12]:
# Estrutura final
prior_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 15 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   order_id                int32   
 1   product_id              int32   
 2   add_to_cart_order       int64   
 3   reordered               int64   
 4   user_id                 int32   
 5   eval_set                object  
 6   order_number            int64   
 7   order_dow               int64   
 8   order_hour_of_day       int64   
 9   days_since_prior_order  float64 
 10  product_name            object  
 11  aisle_id                int64   
 12  department_id           int64   
 13  aisle                   category
 14  department              category
dtypes: category(2), float64(1), int32(3), int64(7), object(2)
memory usage: 2.9+ GB


In [13]:
import pandas as pd
import pyarrow as pa

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Pandas: 2.3.1
PyArrow: 21.0.0


In [ ]:
# Exportando arquivo para continuar na etapa de EDA
prior_full.to_parquet(
    '../data/instacart_orders.parquet',
    index=False
)